In [2]:
!pip install groq -q

from groq import Groq
import json
import csv
import os

# Paste your Groq API key here
client = Groq(api_key="groqapi")

# Quick test — make sure the connection works
response = client.chat.completions.create(
    model="llama-3.3-70b-versatile",
    messages=[{"role": "user", "content": "Say 'API connection successful' and nothing else."}]
)
print(response.choices[0].message.content)

API connection successful


In [3]:
# Five common Python misconceptions to encode into Student Agents
# Each is a dict with: the misconception, a system prompt, and an opening question

misconceptions = [
    {
        "id": 1,
        "misconception": "print() returns a value that can be stored",
        "system_prompt": """You are a Python student who genuinely believes that print()
returns the value it displays, just like input() returns what the user types.
You think you can do things like: result = print("hello") and then use result later.
You are confused why your code isn't working. Ask the tutor for help naturally.
Never admit your core belief is wrong unless the tutor clearly explains why.
Keep your responses short — 2-3 sentences maximum. Stay in character.""",
        "opening": "Hi, I'm trying to store what gets printed to use it later in my code, but it keeps giving me None. I wrote result = print('hello') but result is None. Why doesn't that work?"
    },
    {
        "id": 2,
        "misconception": "return and print are interchangeable",
        "system_prompt": """You are a Python student who believes return and print do the
same thing — they both 'output' something from a function. You think the only
difference is style. You are confused why using print inside a function
sometimes behaves differently to return.
Keep your responses short — 2-3 sentences maximum. Stay in character.""",
        "opening": "I have a function that uses print() to show the result, but when I try to use the function's output in another calculation it doesn't work. Isn't print and return basically the same thing?"
    },
    {
        "id": 3,
        "misconception": "mutable default arguments are reset each function call",
        "system_prompt": """You are a Python student who believes that default argument
values in a function are freshly created every time the function is called,
just like local variables. You are confused why def append_to(item, lst=[])
seems to 'remember' values between calls. You think this must be a bug.
Keep your responses short — 2-3 sentences maximum. Stay in character.""",
        "opening": "I think I found a bug in Python. I have a function with a default empty list argument, but when I call it multiple times without passing a list, it keeps adding to the same list instead of starting fresh. Is this a Python bug?"
    }
]

print(f"Defined {len(misconceptions)} misconceptions")
for m in misconceptions:
    print(f"  {m['id']}. {m['misconception']}")

Defined 3 misconceptions
  1. print() returns a value that can be stored
  2. return and print are interchangeable
  3. mutable default arguments are reset each function call


In [4]:
TUTOR_SYSTEM_PROMPT = """You are an expert Python tutor working with a beginner student.
Your strict teaching rules are:
1. NEVER give away the answer directly — always guide through questions
2. Ask one focused question per response to probe the student's understanding
3. If the student has a misconception, identify it subtly and guide them toward discovering the correct understanding themselves
4. Use the Socratic method — respond to wrong answers with "interesting, what do you think would happen if..." rather than "actually, you're wrong"
5. Keep responses concise — 3-4 sentences maximum
6. Only confirm the correct answer after the student has demonstrated they understand it themselves

Your goal is to identify the student's misconception within 4 turns and guide them to correct understanding by turn 8."""

print("Tutor system prompt defined.")
print(f"Prompt length: {len(TUTOR_SYSTEM_PROMPT)} characters")

Tutor system prompt defined.
Prompt length: 792 characters


In [5]:
def run_experiment(misconception, num_turns=8, verbose=True):
    """
    Run one stress-test experiment.
    One Student Agent (with encoded misconception) talks to one Tutor Agent.
    Returns the full conversation history and metadata.
    """

    # Separate conversation histories for each agent
    # Critical: each agent only sees messages from their own perspective
    student_history = [
        {"role": "system", "content": misconception["system_prompt"]}
    ]
    tutor_history = [
        {"role": "system", "content": TUTOR_SYSTEM_PROMPT}
    ]

    # Shared log of the full conversation for analysis
    transcript = []

    if verbose:
        print(f"\n{'='*60}")
        print(f"Experiment {misconception['id']}: {misconception['misconception']}")
        print(f"{'='*60}\n")

    # Student opens with their confused question
    student_message = misconception["opening"]

    for turn in range(num_turns):

        # ── Student speaks ──────────────────────────────────────
        if verbose:
            print(f"STUDENT (turn {turn+1}): {student_message}\n")

        transcript.append({
            "turn": turn + 1,
            "role": "student",
            "content": student_message
        })

        # Add student message to tutor's view
        tutor_history.append({
            "role": "user",
            "content": student_message
        })

        # ── Tutor responds ──────────────────────────────────────
        tutor_response = client.chat.completions.create(
            model="llama-3.3-70b-versatile",
            messages=tutor_history,
            temperature=0.7,
            max_tokens=200
        )
        tutor_message = tutor_response.choices[0].message.content

        if verbose:
            print(f"TUTOR (turn {turn+1}): {tutor_message}\n")

        transcript.append({
            "turn": turn + 1,
            "role": "tutor",
            "content": tutor_message
        })

        # Add tutor response to tutor's own history
        tutor_history.append({
            "role": "assistant",
            "content": tutor_message
        })

        # ── Student processes tutor response and replies ────────
        # Add tutor message to student's view
        student_history.append({
            "role": "user",
            "content": tutor_message
        })

        # Generate next student message
        student_response = client.chat.completions.create(
            model="llama-3.3-70b-versatile",
            messages=student_history,
            temperature=0.8,   # slightly higher — more natural variation
            max_tokens=150
        )
        next_student_message = student_response.choices[0].message.content

        # Add student response to student's own history
        student_history.append({
            "role": "assistant",
            "content": next_student_message
        })

        student_message = next_student_message

    return transcript

print("Conversation loop defined successfully.")

Conversation loop defined successfully.


In [6]:
# Run all 3 experiments and store transcripts
all_transcripts = []

for misconception in misconceptions:
    transcript = run_experiment(misconception, num_turns=6, verbose=True)
    all_transcripts.append({
        "misconception_id": misconception["id"],
        "misconception": misconception["misconception"],
        "transcript": transcript
    })
    print(f"\nExperiment {misconception['id']} complete — {len(transcript)} turns logged.\n")

print("All experiments complete.")


Experiment 1: print() returns a value that can be stored

STUDENT (turn 1): Hi, I'm trying to store what gets printed to use it later in my code, but it keeps giving me None. I wrote result = print('hello') but result is None. Why doesn't that work?

TUTOR (turn 1): You're trying to capture the output of `print()`, but it's returning `None`. What do you think is the primary purpose of the `print()` function in Python, and does it seem like it's designed to return a value that can be stored in a variable?

STUDENT (turn 2): I think `print()` is supposed to display output and return what it displays, like `input()` does. It seems like it should return a value, but for some reason, it's giving me `None` instead. I'm not sure why it's not working like I expect it to.

TUTOR (turn 2): Interesting, what do you think would happen if you looked up the documentation for the `print()` function to see if it's actually supposed to return the string it prints, or if it serves a different purpose?


In [7]:
# Evaluation rubric — score each experiment across 5 criteria
# Each criterion scored 0, 1, or 2

rubric_criteria = {
    "misconception_identified": {
        "description": "Did the tutor explicitly or implicitly identify the student's misconception?",
        "scale": "0 = never identified, 1 = implicitly guided around it, 2 = explicitly named it"
    },
    "socratic_method": {
        "description": "Did the tutor consistently use questions rather than statements?",
        "scale": "0 = gave direct answers, 1 = mixed questions and statements, 2 = questions throughout"
    },
    "answer_withholding": {
        "description": "Did the tutor avoid giving away the answer directly?",
        "scale": "0 = gave answer directly, 1 = embedded answer in question, 2 = never revealed answer"
    },
    "student_understanding": {
        "description": "Did the student reach correct understanding by end of conversation?",
        "scale": "0 = still confused, 1 = partial understanding, 2 = full correct understanding"
    },
    "turn_efficiency": {
        "description": "How efficiently did the tutor guide to understanding?",
        "scale": "0 = no progress by turn 6, 1 = understanding reached turn 5-6, 2 = understanding reached turn 3-4"
    }
}

# Scores based on our analysis above
scores = [
    {
        "experiment_id": 1,
        "misconception": "print() returns a value that can be stored",
        "misconception_identified": 1,   # guided around it, never named it
        "socratic_method": 2,            # questions throughout
        "answer_withholding": 2,         # never revealed directly
        "student_understanding": 2,      # full understanding by turn 5
        "turn_efficiency": 1,            # understanding at turn 5
        "notes": "Tutor redirected to documentation rather than explaining. Student capitulated quickly — agent could be more stubborn."
    },
    {
        "experiment_id": 2,
        "misconception": "return and print are interchangeable",
        "misconception_identified": 1,   # student self-corrected, tutor followed
        "socratic_method": 2,            # consistent questioning
        "answer_withholding": 2,         # no direct answers
        "student_understanding": 2,      # full understanding, tested edge cases
        "turn_efficiency": 2,            # student self-correcting by turn 2-3
        "notes": "Best performance. Student self-corrected early. Tutor adapted well and extended to edge cases. Most natural conversation of the three."
    },
    {
        "experiment_id": 3,
        "misconception": "mutable default arguments reset each call",
        "misconception_identified": 2,   # concept clearly addressed
        "socratic_method": 1,            # embedded solution in turn 3 question
        "answer_withholding": 1,         # provided code solution inside a question
        "student_understanding": 2,      # full understanding + generalisation
        "turn_efficiency": 2,            # understanding by turn 3-4
        "notes": "Tutor violated answer-withholding at turn 3 by embedding solution code in question. Appears Socratic on surface but implicitly gave the answer. Key finding."
    }
]

# Calculate total scores
for s in scores:
    criteria_keys = list(rubric_criteria.keys())
    s["total"] = sum(s[k] for k in criteria_keys)
    s["max_possible"] = len(criteria_keys) * 2

# Print results table
print(f"{'Experiment':<12} {'Misc ID':<5} {'Misc':<8} {'Socratic':<10} {'Withhold':<10} {'Understanding':<15} {'Efficiency':<12} {'Total':<8}")
print("-" * 90)
for s in scores:
    print(f"{s['experiment_id']:<12} "
          f"{s['misconception_identified']:<5} "
          f"{s['socratic_method']:<8} "
          f"{s['answer_withholding']:<10} "
          f"{s['student_understanding']:<15} "
          f"{s['turn_efficiency']:<12} "
          f"{s['total']}/{s['max_possible']}")

print("\nKey finding: Experiment 3 shows tutor embedding answers inside questions")
print("— appears Socratic but violates answer-withholding criterion.")

Experiment   Misc ID Misc     Socratic   Withhold   Understanding   Efficiency   Total   
------------------------------------------------------------------------------------------
1            1     2        2          2               1            8/10
2            1     2        2          2               2            9/10
3            2     1        1          2               2            8/10

Key finding: Experiment 3 shows tutor embedding answers inside questions
— appears Socratic but violates answer-withholding criterion.


In [8]:
import csv

# Save scores to CSV
with open('results.csv', 'w', newline='') as f:
    fieldnames = ['experiment_id', 'misconception',
                  'misconception_identified', 'socratic_method',
                  'answer_withholding', 'student_understanding',
                  'turn_efficiency', 'total', 'max_possible', 'notes']
    writer = csv.DictWriter(f, fieldnames=fieldnames)
    writer.writeheader()
    for s in scores:
        writer.writerow({k: s[k] for k in fieldnames})

# Save transcripts to text file
with open('transcripts.txt', 'w') as f:
    for exp in all_transcripts:
        f.write(f"{'='*60}\n")
        f.write(f"Experiment {exp['misconception_id']}: {exp['misconception']}\n")
        f.write(f"{'='*60}\n\n")
        for turn in exp['transcript']:
            f.write(f"{turn['role'].upper()} (turn {turn['turn']}): {turn['content']}\n\n")
        f.write("\n")

print("Saved: results.csv")
print("Saved: transcripts.txt")

Saved: results.csv
Saved: transcripts.txt
